In [7]:
# Colab dependencies
!pip -q install -U pytorch-lightning transformers datasets evaluate jiwer librosa soundfile


In [8]:
from google.colab import drive
import os
import glob

drive.mount('/content/drive', force_remount=True)

DATA_ROOT = '/content/drive/MyDrive/audio lab 3'
LABELS_PATH = '/content/drive/MyDrive/audio lab 3/labels.jsonl'
OUTPUT_DIR = '/content/drive/MyDrive/audio lab 3/whisper_toronto_ckpts'
os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.exists(LABELS_PATH):
    candidates = (
        glob.glob(os.path.join(DATA_ROOT, '**', 'labels.jsonl'), recursive=True)
        + glob.glob(os.path.join(DATA_ROOT, '**', 'labels.json'), recursive=True)
    )
    if candidates:
        LABELS_PATH = candidates[0]

print('DATA_ROOT =', DATA_ROOT)
print('LABELS_PATH =', LABELS_PATH)
print('OUTPUT_DIR =', OUTPUT_DIR)

Mounted at /content/drive
DATA_ROOT = /content/drive/MyDrive/audio lab 3
LABELS_PATH = /content/drive/MyDrive/audio lab 3/toronto dataset/labels.jsonl
OUTPUT_DIR = /content/drive/MyDrive/audio lab 3/whisper_toronto_ckpts


In [9]:
import os
import re
import json
import glob
import random
import numpy as np
import pandas as pd
import torch
import librosa
import pytorch_lightning as pl

from torch.utils.data import DataLoader
from datasets import Dataset, concatenate_datasets, load_dataset
import evaluate
from transformers import WhisperForConditionalGeneration, WhisperProcessor, get_linear_schedule_with_warmup

pl.seed_everything(42, workers=True)
random.seed(42)
np.random.seed(42)
torch.set_float32_matmul_precision('medium')

TEST_LINES = [
    'toronto_27', 'toronto_46', 'toronto_42', 'toronto_37', 'toronto_89',
    'toronto_43', 'toronto_157', 'toronto_9', 'toronto_156', 'toronto_7',
    'toronto_123', 'toronto_54', 'toronto_67', 'toronto_62', 'toronto_81',
    'toronto_134', 'toronto_148', 'toronto_21', 'toronto_135', 'toronto_166',
    'toronto_58'
]


def make_config():
    return {
        'model_id': 'openai/whisper-small',
        'language': 'ukrainian',
        'task': 'transcribe',
        'sample_rate': 16000,

        'batch_size': 8,
        'num_workers': 2,
        'lr': 1e-5,
        'weight_decay': 1e-2,
        'warmup_ratio': 0.1,
        'max_epochs': 5,
        'grad_accum': 1,
        'val_size': 0.1,
        'max_gen_tokens': 128,
        'seed': 42,

        # Set limits if you want smaller subsets; use None for full split
        'max_train_samples': 3000,
        'max_val_samples': 400,
        'max_test_samples': 500,

        'toronto_root_dir': DATA_ROOT,
        'toronto_labels_path': LABELS_PATH,

        'run_integrity_check': True,
        'show_empty_parts': False,
        'show_partial_parts': False,
        'max_parts_to_show': 200,

        'extra_datasets': [],

        'output_dir': OUTPUT_DIR,
    }


def extract_line_id(path_like):
    p = str(path_like).replace('\\', '/')
    m = re.search(r'(toronto_\d+)', p)
    if m:
        return m.group(1)
    stem = os.path.splitext(os.path.basename(p))[0]
    m = re.match(r'(toronto_\d+)_\d+$', stem)
    if m:
        return m.group(1)
    return stem


def load_labels_mapping(labels_path):
    with open(labels_path, 'r', encoding='utf-8') as f:
        raw = f.read().strip()

    try:
        obj = json.loads(raw)
        if isinstance(obj, dict):
            return obj
    except json.JSONDecodeError:
        pass

    mapping = {}
    for line in raw.splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            row = json.loads(line)
        except json.JSONDecodeError:
            continue

        if isinstance(row, dict):
            if 'audio' in row and 'text' in row:
                mapping[row['audio']] = row['text']
            elif len(row) == 1:
                k, v = next(iter(row.items()))
                mapping[k] = v
    return mapping


def build_audio_index(root_dir):
    wav_files = glob.glob(os.path.join(root_dir, '**', '*.wav'), recursive=True)
    by_basename = {}
    for p in wav_files:
        by_basename.setdefault(os.path.basename(p), []).append(p)
    return by_basename


def resolve_audio_path(raw_path, root_dir, by_basename):
    raw_path = str(raw_path).replace('\\', '/').lstrip('./')
    candidates = [raw_path]
    if raw_path.startswith('dataset/'):
        candidates.append(raw_path[len('dataset/'):])
    candidates.append(os.path.basename(raw_path))

    seen = set()
    for rel in candidates:
        rel = rel.replace('/', os.sep)
        if rel in seen:
            continue
        seen.add(rel)
        abs_p = os.path.join(root_dir, rel)
        if os.path.exists(abs_p):
            return abs_p

    by_name = by_basename.get(os.path.basename(raw_path), [])
    if by_name:
        return by_name[0]
    return None


def build_expected_parts(labels_map):
    expected = {}
    for audio_key in labels_map.keys():
        raw = str(audio_key).replace('\\', '/').lstrip('./')
        part = extract_line_id(raw)

        m = re.search(r'(toronto_\d+)/([^/]+\.wav)$', raw)
        if m:
            part = m.group(1)
            wav_name = m.group(2)
        else:
            wav_name = os.path.basename(raw)

        if not wav_name.lower().endswith('.wav'):
            continue

        expected.setdefault(part, set()).add(wav_name)
    return expected


def build_existing_parts(root_dir):
    existing = {}
    part_dirs = [p for p in glob.glob(os.path.join(root_dir, 'toronto_*')) if os.path.isdir(p)]
    for d in part_dirs:
        part = os.path.basename(d)
        wavs = glob.glob(os.path.join(d, '**', '*.wav'), recursive=True)
        existing[part] = {os.path.basename(p) for p in wavs}
    return existing


def analyze_dataset_integrity(root_dir, labels_map, cfg):
    expected = build_expected_parts(labels_map)
    existing = build_existing_parts(root_dir)

    all_parts = sorted(set(expected.keys()) | set(existing.keys()))

    empty_parts = []
    partial_parts = []
    complete_parts = []

    for part in all_parts:
        expected_files = expected.get(part, set())
        if not expected_files:
            continue

        existing_files = existing.get(part, set())
        matched = expected_files & existing_files

        expected_count = len(expected_files)
        existing_count = len(existing_files)
        matched_count = len(matched)

        if matched_count == 0:
            empty_parts.append((part, expected_count, existing_count))
        elif matched_count < expected_count:
            partial_parts.append((part, matched_count, expected_count, existing_count))
        else:
            complete_parts.append((part, expected_count, existing_count))

    print(
        'Integrity summary:',
        f'expected_parts={len(expected)} | complete={len(complete_parts)} | '
        f'partial={len(partial_parts)} | empty={len(empty_parts)}'
    )

    if cfg.get('show_empty_parts', False):
        print('Empty parts:')
        for part, exp_cnt, exist_cnt in empty_parts[:cfg.get('max_parts_to_show', 200)]:
            print(f'  {part} (matched=0, expected={exp_cnt}, files_found={exist_cnt})')

    if cfg.get('show_partial_parts', False):
        print('Partial parts:')
        for part, matched_cnt, exp_cnt, exist_cnt in partial_parts[:cfg.get('max_parts_to_show', 200)]:
            print(f'  {part} (matched={matched_cnt}/{exp_cnt}, files_found={exist_cnt})')

    return {
        'expected_parts': len(expected),
        'complete_parts': len(complete_parts),
        'partial_parts': len(partial_parts),
        'empty_parts': len(empty_parts),
        'empty_part_names': [x[0] for x in empty_parts],
        'partial_part_names': [x[0] for x in partial_parts],
    }


def load_toronto_dataset(cfg):
    if not os.path.exists(cfg['toronto_labels_path']):
        raise FileNotFoundError(f"labels file not found: {cfg['toronto_labels_path']}")

    root_dir = cfg['toronto_root_dir']
    if not os.path.isdir(root_dir):
        root_dir = os.path.dirname(cfg['toronto_labels_path'])

    labels_map = load_labels_mapping(cfg['toronto_labels_path'])

    if cfg.get('run_integrity_check', True):
        _ = analyze_dataset_integrity(root_dir, labels_map, cfg)

    by_basename = build_audio_index(root_dir)

    rows = []
    dropped = 0
    for audio_key, text in labels_map.items():
        if text is None or str(text).strip() == '':
            continue

        abs_audio = resolve_audio_path(audio_key, root_dir, by_basename)
        if not abs_audio:
            dropped += 1
            continue

        rows.append({
            'audio_path': abs_audio,
            'text': str(text).strip(),
            'line_id': extract_line_id(audio_key),
        })

    if not rows:
        raise RuntimeError('No usable rows found in labels.')

    if dropped > 0:
        print(f'Warning: dropped {dropped} rows due to missing audio files.')

    ds = Dataset.from_pandas(pd.DataFrame(rows), preserve_index=False)
    ds = ds.filter(lambda x: x['text'] is not None and str(x['text']).strip() != '')
    return ds


def normalize_extra_dataset(ds, spec):
    audio_col = spec.get('audio_col', 'audio')
    text_col = spec.get('text_col', 'text')
    id_col = spec.get('id_col')

    if audio_col != 'audio':
        ds = ds.rename_column(audio_col, 'audio')
    if text_col != 'text':
        ds = ds.rename_column(text_col, 'text')
    if id_col and id_col in ds.column_names and id_col != 'line_id':
        ds = ds.rename_column(id_col, 'line_id')
    if 'line_id' not in ds.column_names:
        ds = ds.add_column('line_id', [f'extra_{i}' for i in range(len(ds))])

    keep = [c for c in ['audio', 'text', 'line_id'] if c in ds.column_names]
    ds = ds.select_columns(keep)

    def to_path(batch):
        out = []
        for a in batch['audio']:
            if isinstance(a, dict) and a.get('path'):
                out.append(a['path'])
            elif isinstance(a, str):
                out.append(a)
            else:
                out.append(None)
        batch['audio_path'] = out
        return batch

    ds = ds.map(to_path, batched=True)
    ds = ds.filter(lambda x: x['audio_path'] is not None and str(x['audio_path']).strip() != '')
    ds = ds.select_columns(['audio_path', 'text', 'line_id'])
    return ds


def load_extra_train_datasets(cfg):
    out = []
    for spec in cfg.get('extra_datasets', []):
        ds = load_dataset(spec['path'], spec.get('name'), split=spec.get('split', 'train'))
        ds = normalize_extra_dataset(ds, spec)
        out.append(ds)
    return out


def take_n_samples(ds, n, seed, name):
    if n is None:
        return ds
    if n <= 0:
        return ds.select([])
    if n >= len(ds):
        if n > len(ds):
            print(f'Warning: requested {name}={n}, but only {len(ds)} available; using all available.')
        return ds
    return ds.shuffle(seed=seed).select(range(n))


def make_train_val_split(train_val_ds, cfg):
    seed = cfg.get('seed', 42)
    train_n = cfg.get('max_train_samples')
    val_n = cfg.get('max_val_samples')

    if train_n is not None and val_n is not None:
        shuffled = train_val_ds.shuffle(seed=seed)

        val_take = max(0, min(val_n, len(shuffled)))
        remaining = max(0, len(shuffled) - val_take)
        train_take = max(0, min(train_n, remaining))

        if val_take < val_n:
            print(f'Warning: requested val={val_n}, but only {val_take} available after split constraints.')
        if train_take < train_n:
            print(f'Warning: requested train={train_n}, but only {train_take} available after val allocation.')

        val_ds = shuffled.select(range(val_take))
        train_ds = shuffled.select(range(val_take, val_take + train_take))
        return train_ds, val_ds

    split = train_val_ds.train_test_split(test_size=cfg['val_size'], seed=seed)
    train_ds = take_n_samples(split['train'], train_n, seed, 'max_train_samples')
    val_ds = take_n_samples(split['test'], val_n, seed, 'max_val_samples')
    return train_ds, val_ds


CFG = make_config()

toronto_ds = load_toronto_dataset(CFG)
test_set = set(TEST_LINES)

test_ds = toronto_ds.filter(lambda line_id: line_id in test_set, input_columns=['line_id'])
train_val_ds = toronto_ds.filter(lambda line_id: line_id not in test_set, input_columns=['line_id'])

if len(test_ds) == 0:
    raise RuntimeError('test_ds is empty. Check line_id extraction from your labels keys.')

present_test_ids = set(test_ds['line_id'])
missing_test_ids = sorted(list(test_set - present_test_ids))
if missing_test_ids:
    print('Warning: missing test line_ids in your local dataset copy:', missing_test_ids)

test_ds = take_n_samples(test_ds, CFG.get('max_test_samples'), CFG.get('seed', 42), 'max_test_samples')
train_ds, val_ds = make_train_val_split(train_val_ds, CFG)

extra_train = load_extra_train_datasets(CFG)
if extra_train:
    train_ds = concatenate_datasets([train_ds] + extra_train)
    train_ds = take_n_samples(train_ds, CFG.get('max_train_samples'), CFG.get('seed', 42), 'max_train_samples')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")





INFO:lightning_fabric.utilities.seed:Seed set to 42


Integrity summary: expected_parts=114 | complete=0 | partial=0 | empty=114


Filter:   0%|          | 0/2538 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2538 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2538 [00:00<?, ? examples/s]

Device: cuda
Train: 1607 | Val: 400 | Test: 500


In [10]:
processor = WhisperProcessor.from_pretrained(
    CFG['model_id'],
    language=CFG['language'],
    task=CFG['task']
)


def load_audio_16k(path, target_sr):
    audio, sr = librosa.load(path, sr=target_sr, mono=True)
    return audio


def make_data_collator(processor, cfg):
    def collate(features):
        arrays = [load_audio_16k(f['audio_path'], cfg['sample_rate']) for f in features]
        texts = [f['text'] for f in features]

        feat_batch = processor.feature_extractor(
            arrays,
            sampling_rate=cfg['sample_rate'],
            return_tensors='pt',
            padding='max_length',
            max_length=processor.feature_extractor.n_samples,  # 480000
            truncation=True,
        )

        tokenized = processor.tokenizer(
            texts,
            return_tensors='pt',
            padding=True,
            truncation=True,
        )
        labels = tokenized.input_ids.masked_fill(tokenized.attention_mask.ne(1), -100)

        bos = processor.tokenizer.bos_token_id
        if labels.size(1) > 0 and (labels[:, 0] == bos).all():
            labels = labels[:, 1:]

        return {
            'input_features': feat_batch.input_features,
            'labels': labels,
        }

    return collate


data_collator = make_data_collator(processor, CFG)

In [11]:
def make_train_dataloader(train_dataset, collator, cfg):
    return DataLoader(
        train_dataset,
        batch_size=cfg['batch_size'],
        shuffle=True,
        collate_fn=collator,
        num_workers=cfg['num_workers'],
        pin_memory=torch.cuda.is_available(),
    )


def make_val_dataloader(val_dataset, collator, cfg):
    return DataLoader(
        val_dataset,
        batch_size=cfg['batch_size'],
        shuffle=False,
        collate_fn=collator,
        num_workers=cfg['num_workers'],
        pin_memory=torch.cuda.is_available(),
    )


class WhisperLightningModule(pl.LightningModule):
    def __init__(self, cfg, processor):
        super().__init__()
        self.cfg = cfg
        self.processor = processor
        self.model = WhisperForConditionalGeneration.from_pretrained(cfg['model_id'])
        self.model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(
            language=cfg['language'],
            task=cfg['task']
        )
        self.model.config.suppress_tokens = []

        self.wer_metric = evaluate.load('wer')
        self.cer_metric = evaluate.load('cer')
        self.val_preds = []
        self.val_refs = []

    def training_step(self, batch, batch_idx):
        out = self.model(input_features=batch['input_features'], labels=batch['labels'])
        self.log('train_loss', out.loss, prog_bar=True, on_step=True, on_epoch=True, batch_size=batch['input_features'].size(0))
        return out.loss

    def validation_step(self, batch, batch_idx):
        out = self.model(input_features=batch['input_features'], labels=batch['labels'])
        self.log('val_loss', out.loss, prog_bar=True, on_epoch=True, batch_size=batch['input_features'].size(0))

        gen_ids = self.model.generate(batch['input_features'], max_new_tokens=self.cfg['max_gen_tokens'])
        preds = self.processor.batch_decode(gen_ids, skip_special_tokens=True)

        labels = batch['labels'].detach().clone()
        labels[labels == -100] = self.processor.tokenizer.pad_token_id
        refs = self.processor.batch_decode(labels, skip_special_tokens=True)

        self.val_preds.extend(preds)
        self.val_refs.extend(refs)

    def on_validation_epoch_end(self):
        if len(self.val_refs) > 0:
            wer = self.wer_metric.compute(predictions=self.val_preds, references=self.val_refs)
            cer = self.cer_metric.compute(predictions=self.val_preds, references=self.val_refs)
            self.log('val_wer', wer, prog_bar=True)
            self.log('val_cer', cer, prog_bar=True)
        self.val_preds = []
        self.val_refs = []

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.cfg['lr'], weight_decay=self.cfg['weight_decay'])
        total_steps = self.trainer.estimated_stepping_batches
        warmup_steps = int(total_steps * self.cfg['warmup_ratio'])
        scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
        return [optimizer], [{'scheduler': scheduler, 'interval': 'step'}]


In [12]:
train_loader = make_train_dataloader(train_ds, data_collator, CFG)
val_loader = make_val_dataloader(val_ds, data_collator, CFG)

lit_model = WhisperLightningModule(CFG, processor)

ckpt_cb = pl.callbacks.ModelCheckpoint(
    dirpath=CFG['output_dir'],
    monitor='val_wer',
    mode='min',
    save_top_k=1,
    filename='whisper-toronto-{epoch:02d}-{val_wer:.4f}'
)

trainer = pl.Trainer(
    max_epochs=CFG['max_epochs'],
    accelerator='gpu' if torch.cuda.is_available() else 'cpu',
    devices=1,
    precision='16-mixed' if torch.cuda.is_available() else 32,
    accumulate_grad_batches=CFG['grad_accum'],
    gradient_clip_val=1.0,
    log_every_n_steps=10,
    enable_progress_bar=True,
    callbacks=[ckpt_cb, pl.callbacks.TQDMProgressBar(refresh_rate=1)],
)

trainer.fit(lit_model, train_dataloaders=train_loader, val_dataloaders=val_loader)
print('Best checkpoint:', ckpt_cb.best_model_path)



Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.utilities.rank_zero:Loading `train_dataloader` to estimate number of stepping batches.


┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type                            ┃ Params ┃ Mode ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━╇━━━━━━━┩
│ 0 │ model │ WhisperForConditionalGeneration │  241 M │ eval │     0 │
└───┴───────┴─────────────────────────────────┴────────┴──────┴───────┘

Trainable params: 241 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 241 M                                                                                                
Total estimated model params size (MB): 966                                                                        
Modules in train mode: 0                                                                                           
Modules in eval mode: 350                                                                                          
Total FLOPs: 0

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

Validation: |          | 0/? [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

Best checkpoint: /content/drive/.shortcut-targets-by-id/1LkVqCSHYGOWkCKfYs4ULB_HRRi-dAnrF/audio lab 3/whisper_toronto_ckpts/whisper-toronto-epoch=03-val_wer=0.3380.ckpt


In [14]:
def eval_collate(samples):
    arrays = [load_audio_16k(s['audio_path'], CFG['sample_rate']) for s in samples]
    feats = processor.feature_extractor(
        arrays,
        sampling_rate=CFG['sample_rate'],
        return_tensors='pt',
        padding='max_length',
        max_length=processor.feature_extractor.n_samples,
        truncation=True,
    )
    return {
        'input_features': feats.input_features,
        'refs': [s['text'] for s in samples],
        'line_ids': [s['line_id'] for s in samples],
    }


def evaluate_on_test(model_module, test_dataset, batch_size=8):
    model = model_module.model.to(device)
    model.eval()

    loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=eval_collate)
    preds, refs, ids = [], [], []

    for batch in loader:
        in_feats = batch['input_features'].to(device)
        with torch.no_grad():
            gen_ids = model.generate(in_feats, max_new_tokens=CFG['max_gen_tokens'])
        p = processor.batch_decode(gen_ids, skip_special_tokens=True)

        preds.extend(p)
        refs.extend(batch['refs'])
        ids.extend(batch['line_ids'])

    wer = evaluate.load('wer').compute(predictions=preds, references=refs)
    cer = evaluate.load('cer').compute(predictions=preds, references=refs)
    return wer, cer, pd.DataFrame({'line_id': ids, 'reference': refs, 'prediction': preds})


best_path = ckpt_cb.best_model_path
if best_path:
    best_model = WhisperLightningModule.load_from_checkpoint(best_path, cfg=CFG, processor=processor)
else:
    best_model = lit_model


test_wer, test_cer, pred_df = evaluate_on_test(best_model, test_ds, batch_size=CFG['batch_size'])
print(f'TEST WER: {test_wer:.4f}')
print(f'TEST CER: {test_cer:.4f}')
pred_df.head(10)


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

TEST WER: 0.3524
TEST CER: 0.0981


,line_id,reference,prediction
0,toronto_9,"А ще ці езотеричні некрокомуністи впевнені, що...","А ще ці езотаричні некрокомоністи – впевнення,..."
1,toronto_7,в поліцейському відділку в Кагарлику. Рінат Ах...,у поліцейському відділку в Ка́герле́ко. Рінат ...
2,toronto_7,"Дякую, що визнаєте мою здатність продовжувати ...","Дякую, що ви знаєте мою здатність продовжувати..."
3,toronto_7,що не лише Шустер алегорично робить мінет Ахме...,"що не лише шустер, але горично робить мінет Ах..."
4,toronto_9,на якому рясно розквіт антивакцинаторський рух...,на якому рясно розквіт антивакцинаторський рух...
5,toronto_7,Мета – не дати Україні увійти до НАТО.,мета не дати Україні увіти доната.
6,toronto_9,"Тож, і терпи. Завдяки цьому тренду Поплавський...","Тож і терпи. Завдяки цьому тренду, Поплавський..."
7,toronto_9,"Цей інцидент демонструє, що Росія зараз розроб...","Цей інцидент демонструє, що Росія зараз розроб..."
8,toronto_7,І за підсумками дня народження Шустера тепер і...,І за підсумками дня народження Шустера. Тепері...
9,toronto_9,"Когда ты проходишь мимо, милая… Облака наверня...","Когда ти проходиш мім маммилая. Аблака, навірн..."
